# 01 — Data Loading and Cleaning

Loads and cleans five MHCLG Live Tables, producing tidy long-format datasets in `data/processed/` and a SQLite database.

**Data source:** MHCLG Live Tables on housing supply (net additional dwellings, house building indicators, affordable housing supply)

## Why a separate loading notebook?

I keep loading, exploration, and analysis in separate notebooks. The loading step is messy by nature — it deals with file quirks, bad column names, and missing value codes. Once it runs, every downstream notebook reads from clean CSVs and never touches the raw files again.

This is also the notebook that documents the data pipeline: what came in, what decisions I made, what came out.

## The five tables

| Table | File | What it contains |
|---|---|---|
| 122 | Live_Table_122.ods | Net additional dwellings by local authority, 2001–2025 |
| 244 | Live_Table_244.xlsx | England-level starts and completions, 1946–2025 |
| 253 | Live_Table_253.xlsx | District-level starts and completions, 1980–2025 |
| 118 | Live_Table_118.ods | Regional net additions with component breakdown |
| 1006–1008 | Live_Tables_1006_to_1008_Completions.ods | LA-level affordable completions by tenure |

## Imports

In [1]:
import re
import sqlite3
from pathlib import Path

import pandas as pd

RAW = Path('../data/raw')
PROCESSED = Path('../data/processed')
PROCESSED.mkdir(parents=True, exist_ok=True)
DB_PATH = PROCESSED / 'housing.db'

print('pandas:', pd.__version__)
print('\nRaw files:')
for f in sorted(RAW.iterdir()):
    print(f'  {f.name}  ({f.stat().st_size // 1024} KB)')

pandas: 2.3.3

Raw files:
  Live_Table_1000.ods  (29 KB)
  Live_Table_1011.xlsx  (5819 KB)
  Live_Table_118.ods  (49 KB)
  Live_Table_120.ods  (14 KB)
  Live_Table_122.ods  (90 KB)
  Live_Table_244.xlsx  (21 KB)
  Live_Table_253.xlsx  (1168 KB)
  Live_Tables_1006_to_1008_Completions.ods  (324 KB)


## MHCLG file structure

I previewed every raw sheet before writing any cleaning code. Government Excel files almost never have clean row-1 headers. The MHCLG pattern is:

```
Row 1: Table title
Row 2: Dataset description
Row 3: Source citation
Row 4: Additional note (sometimes)
Row 5: Actual column headers   <-- where pandas should start
Row 6+: Data
```

MHCLG also uses bracketed codes instead of blank cells for missing data:

| Code | Meaning |
|---|---|
| `[x]` | Data unavailable (e.g. a UA that didn't exist yet) |
| `[z]` | Not applicable (e.g. an old district merged into a UA) |
| `[p]` | Provisional — may be revised |
| `[r]` | Revised since last publication |

`[p]` and `[r]` appear in column *names* (e.g. `2024-25 [p]`). `[x]` and `[z]` appear in data cells.

## Helper functions

In [2]:
def clean_year_col(col_name: str) -> str:
    # '2024-25 [p]' -> '2024-25'
    return re.sub(r'\s*\[.*?\]', '', str(col_name)).strip()


def replace_markers(df: pd.DataFrame) -> pd.DataFrame:
    # Replace [x], [z] and any other bracketed codes with NaN
    return df.replace(regex={r'^\[.*\]$': float('nan')})


def to_sqlite(df: pd.DataFrame, table_name: str) -> None:
    with sqlite3.connect(DB_PATH) as conn:
        df.to_sql(table_name, conn, if_exists='replace', index=False)

---
## Table 122 — Net additional dwellings by local authority

Net additional dwellings is the headline housing delivery metric for English councils. 'Net' means new builds + conversions + change of use minus demolitions. For Huntingdonshire (mostly suburban/rural) this is very close to the raw new build figure because demolitions are rare.

This is what a council's planning policy team reports on annually. Runs 2001-02 to 2024-25.

**Structure:** Four metadata rows, then the real header, then years as columns. I melt it into long format — one row per authority per year — to make filtering and plotting straightforward.

Raw preview first:

In [3]:
raw_preview = pd.read_excel(
    RAW / 'Live_Table_122.ods',
    sheet_name='LT_122',
    header=None,
    nrows=8,
    engine='odf',
)
raw_preview.iloc[:, :6]

,0,1,2,3,4,5
0,Net additional dwellings by local authority di...,NaN,NaN,NaN,NaN,NaN
1,This worksheet contains 1 table.,NaN,NaN,NaN,NaN,NaN
2,"This table contains notes, which can be found ...",NaN,NaN,NaN,NaN,NaN
3,Source: Housing supply: net additional dwellin...,NaN,NaN,NaN,NaN,NaN
4,DCLG code,Former ONS code,Current ONS code,Authority data,2001-02,2002-03
5,NaN,NaN,E92000001,England,146704,159875
6,NaN,NaN,NaN,Unitary Authorities,25553,27553
7,F0114,00HA,E06000022,Bath and North East Somerset UA,270,264


Row 4 is the actual header. Rows 0–3 are metadata. Loading with `header=4`:

In [4]:
t122_raw = pd.read_excel(
    RAW / 'Live_Table_122.ods',
    sheet_name='LT_122',
    header=4,
    engine='odf',
)

print(f'Shape: {t122_raw.shape}')
print(f'First cols: {list(t122_raw.columns[:4])}')
print(f'Last cols:  {list(t122_raw.columns[-3:])}')

Shape: (421, 28)
First cols: ['DCLG code', 'Former ONS code', 'Current ONS code', 'Authority data']
Last cols:  ['2022-23 [note 16]', '2023-24 [r] [note 17]', '2024-25 [p]']


Last column names have trailing `[p]` and `[r]` annotations. Strip those, replace `[x]` markers, melt to long:

In [5]:
id_cols = ['DCLG code', 'Former ONS code', 'Current ONS code', 'Authority data']

t122_raw.columns = [clean_year_col(c) for c in t122_raw.columns]
t122_raw.columns = id_cols + list(t122_raw.columns[4:])
t122_raw = replace_markers(t122_raw)

t122 = t122_raw.melt(id_vars=id_cols, var_name='year', value_name='net_additions')
t122 = t122.rename(columns={'Current ONS code': 'ons_code', 'Authority data': 'authority_name'})
t122 = t122[['ons_code', 'authority_name', 'year', 'net_additions']]
t122['net_additions'] = pd.to_numeric(t122['net_additions'], errors='coerce')
t122 = t122.dropna(subset=['year'])

print(f'Shape: {t122.shape}  |  Authorities: {t122["ons_code"].nunique()}  |  Years: {t122["year"].min()} to {t122["year"].max()}')
t122[t122['ons_code'] == 'E07000011'].tail(8)

Shape: (10104, 4)  |  Authorities: 417  |  Years: 2001-02 to 2024-25


/var/folders/jf/p4xfj7kx3_l23qw5d2zt9zgm0000gn/T/ipykernel_30382/3022428984.py:8: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df.replace(regex={r'^\[.*\]$': float('nan')})


,ons_code,authority_name,year,net_additions
6894,E07000011,Huntingdonshire,2017-18,837.0
7315,E07000011,Huntingdonshire,2018-19,1131.0
7736,E07000011,Huntingdonshire,2019-20,1101.0
8157,E07000011,Huntingdonshire,2020-21,1132.0
8578,E07000011,Huntingdonshire,2021-22,1055.0
8999,E07000011,Huntingdonshire,2022-23,1250.0
9420,E07000011,Huntingdonshire,2023-24,1239.0
9841,E07000011,Huntingdonshire,2024-25,858.0


---
## Table 244 — England completions and starts, 1946–2025

New build starts and completions for England, split by sector: private enterprise, housing associations, local authorities. Goes back to 1946.

English housing delivery peaked around 1968 at ~352,000 completions and has never been close since. The private/HA/council split tells a big policy story: councils were building as many homes as the private sector in the 1960s–70s, then that collapsed after the 1980s.

**Structure note:** Uses calendar years, not financial years. Something to flag when comparing against Table 122.

In [6]:
t244 = pd.read_excel(RAW / 'Live_Table_244.xlsx', sheet_name='LT_244', header=4, engine='openpyxl')
t244.columns = [str(c).strip() for c in t244.columns]
t244 = t244.rename(columns={'Calendar year': 'year'})

cols = [
    'year',
    'Private Enterprise Starts', 'Housing Association Starts', 'Local Authority Starts', 'All Starts',
    'Private Enterprise Completions', 'Housing Association Completions', 'Local Authority Completions', 'All Completions',
]
t244 = replace_markers(t244[cols].copy())
for c in cols[1:]:
    t244[c] = pd.to_numeric(t244[c], errors='coerce')

t244 = t244.dropna(subset=['year'])
t244['year'] = t244['year'].astype(int)

print(f'Shape: {t244.shape}  |  {t244["year"].min()} to {t244["year"].max()}')
t244.tail(6)

Shape: (80, 9)  |  1946 to 2025


,year,Private Enterprise Starts,Housing Association Starts,Local Authority Starts,All Starts,Private Enterprise Completions,Housing Association Completions,Local Authority Completions,All Completions
74,2020,102450.0,25210.0,1780.0,129440.0,120060,25320,1270,146650
75,2021,142180.0,32410.0,2580.0,177160.0,142130,31220,1590,174940
76,2022,142760.0,37730.0,1580.0,182070.0,144910,31820,1620,178350
77,2023,112860.0,34720.0,3020.0,150600.0,124970,35940,2360,163270
78,2024,76010.0,30820.0,1300.0,108140.0,115690,35380,2780,153850
79,2025,93570.0,30240.0,1040.0,124860.0,104410,35660,1970,142040


`All Starts` is NaN before the mid-1970s — starts data wasn't collected that far back.

The `Local Authority Completions` column is striking: councils were completing over 100,000 homes per year in the late 1960s. By 2000 that had effectively hit zero.

---
## Table 253 — Starts and completions by district, 1980–2025

Same metric as Table 244 but at district level, so I can isolate Huntingdonshire and compare it against neighbours.

**Unusual structure:** Instead of one sheet for all years, this file has 45 separate sheets — `FY_1980_81` through `FY_2024_25`. To build a time series I loop over all 45, extract the data from each, label it with the year, and concatenate.

**Starts vs completions:** A start is when groundwork begins; a completion is when the home is habitable (typically 12–18 months later). If completions are outpacing starts, the pipeline is being drawn down. That's what's happening in Huntingdonshire right now — completions ran ahead of starts in 2023-24 and 2024-25.

In [7]:
t253_xl = pd.ExcelFile(RAW / 'Live_Table_253.xlsx', engine='openpyxl')
fy_sheets = [s for s in t253_xl.sheet_names if s.startswith('FY_')]
print(f'{len(fy_sheets)} year sheets: {fy_sheets[0]} to {fy_sheets[-1]}')

45 year sheets: FY_1980_81 to FY_2024_25


In [8]:
frames = []

for sheet in fy_sheets:
    year_label = sheet.replace('FY_', '').replace('_', '-')
    df = t253_xl.parse(sheet, header=3).iloc[:, :12]
    df.columns = [
        'dclg_code', 'former_ons', 'ons_code', 'authority_name',
        'private_starts', 'ha_starts', 'la_starts', 'all_starts',
        'private_completions', 'ha_completions', 'la_completions', 'all_completions',
    ]
    df = replace_markers(df)
    for col in df.columns[4:]:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df = df[df['ons_code'].notna() & ~df['ons_code'].astype(str).str.startswith('E1')].copy()
    df['year'] = year_label
    frames.append(df[['ons_code', 'authority_name', 'year',
                       'private_starts', 'ha_starts', 'la_starts', 'all_starts',
                       'private_completions', 'ha_completions', 'la_completions', 'all_completions']])

t253 = pd.concat(frames, ignore_index=True)

print(f'Shape: {t253.shape}  |  Authorities: {t253["ons_code"].nunique()}  |  Years: {t253["year"].min()} to {t253["year"].max()}')
t253[t253['ons_code'] == 'E07000011'][['year', 'all_starts', 'all_completions']].tail(8)

Shape: (14553, 11)  |  Authorities: 382  |  Years: 1980-81 to 2024-25


,year,all_starts,all_completions
12192,2017-18,930,630
12518,2018-19,770,850
12844,2019-20,1030,1070
13158,2020-21,680,830
13474,2021-22,1010,890
13783,2022-23,1470,1020
14096,2023-24,910,1440
14392,2024-25,700,920


The pipeline depletion is clear: in 2023-24 completions (1,440) ran well ahead of starts (910). In 2024-25 starts dropped to 700 and completions followed to 920. Worth tracking whether starts recover.

---
## Table 118 — Regional net additions with component breakdown

Breaks net additions down by component — new build, conversions, change of use, demolitions — by region and year. More granular than Table 122 because it shows *what kind* of supply is being delivered.

It also tracks Permitted Development Rights (PDR) separately. PDR lets certain conversions happen without a planning application — office-to-residential, agricultural-to-residential, etc. It bypasses local planning control, which is a live tension for district councils. Understanding how much of regional supply comes through PDR is a genuine policy question.

Two variants in the file: rounded (for publication) and unrounded. I use unrounded.

In [9]:
t118 = pd.read_excel(RAW / 'Live_Table_118.ods', sheet_name='LT118_unrounded', header=4, engine='odf')
t118.columns = [str(c).strip() for c in t118.columns]
t118 = t118.rename(columns={t118.columns[0]: 'component', t118.columns[1]: 'year'})
t118 = replace_markers(t118)

region_cols = [c for c in t118.columns if c not in ('component', 'year', 'Notes')]
t118 = t118[['component', 'year'] + region_cols].copy()
for col in region_cols:
    t118[col] = pd.to_numeric(t118[col], errors='coerce')

t118 = t118.dropna(subset=['year'])
t118 = t118.melt(id_vars=['component', 'year'], var_name='region', value_name='value')

print(f'Shape: {t118.shape}')
print(f'\nComponents:')
for c in t118['component'].unique():
    print(f'  {c}')

Shape: (3170, 4)

Components:
  Net additions
  New build
  Of which under PDR: building upwards to create dwelling houses on detached blocks of flats
  Of which under PDR: building upwards to create dwelling houses on detached commercial or mixed-use buildings
  Of which under PDR: building upwards to create dwelling houses on commercial or mixed-use buildings in a terrace
  Of which under PDR: building upwards to create dwelling houses on dwelling houses in a terrace
  Of which under PDR: building upwards to create dwelling houses on detached dwelling houses
  Of which under PDR: demolition of buildings and construction of dwelling houses
  Of which under PDR: unspecified (new build)
  Of which under PDR: total (new build)
  Net conversions
  Net change of use
  Of which under PDR: agricultural to residential
  Of which under PDR: office to residential
  Of which under PDR: storage to residential
  Of which under PDR: light industrial use to residential
  Of which under PDR: commerci

/var/folders/jf/p4xfj7kx3_l23qw5d2zt9zgm0000gn/T/ipykernel_30382/3022428984.py:8: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df.replace(regex={r'^\[.*\]$': float('nan')})


In [10]:
# East of England net additions — recent trend
(
    t118[(t118['component'] == 'Net additions') & (t118['region'] == 'East of England')]
    .sort_values('year')
    .tail(10)[['year', 'value']]
)

,year,value
1600,2015-16,22495.0
1601,2016-17,25541.0
1602,2017-18,25763.0
1603,2018-19,30189.0
1604,2019-20,28374.0
1605,2020-21,25661.0
1606,2021-22,27457.0
1607,2022-23,30771.0
1608,2023-24,28643.0
1609,2024-25,26012.0


---
## Tables 1006–1008 — Affordable completions by local authority and tenure

**What 'affordable housing' means here:** This is a legal definition, not just 'cheaper than market.' It covers:

| Tenure | Description |
|---|---|
| Social rent | ~50–60% of market rent. Historically the main affordable product; now increasingly replaced by affordable rent. |
| Affordable rent | Up to 80% of market rent. Introduced 2011, now the dominant tenure. Unaffordable in expensive areas. |
| Shared ownership | Buyer purchases a share (25–75%) and pays subsidised rent on the rest. |
| Other | First Homes, rent-to-buy, discount market sale. |

The shift from social rent to affordable rent is politically significant and visible in the data. Councils push for more social rent in Section 106 negotiations; developers push back on viability grounds.

**File structure:** One ODS file, one sheet per tenure type. Header on row 2. `[z]` = not applicable (legacy district merged into a UA).

In [11]:
tenure_sheets = {
    'total_affordable': 'Live_Table_1008C',
    'social_rent':      'Live_Table_1006C',
    'affordable_rent':  'Live_Table_1006aC',
    'shared_ownership': 'Live_Table_1007bC',
}

aff_frames = []
for tenure_label, sheet_name in tenure_sheets.items():
    df = pd.read_excel(
        RAW / 'Live_Tables_1006_to_1008_Completions.ods',
        sheet_name=sheet_name,
        header=2,
        engine='odf',
    )
    df.columns = [str(c).strip() for c in df.columns]
    df = df.rename(columns={
        df.columns[0]: 'region_code', df.columns[1]: 'region_name',
        df.columns[2]: 'ons_code',    df.columns[3]: 'authority_name',
    })
    year_cols = [c for c in df.columns[4:] if re.match(r'\d{4}-\d{2}', c)]
    df = replace_markers(df[['region_code', 'region_name', 'ons_code', 'authority_name'] + year_cols].copy())
    df = df.melt(
        id_vars=['region_code', 'region_name', 'ons_code', 'authority_name'],
        var_name='year', value_name='completions'
    )
    df['tenure'] = tenure_label
    df['completions'] = pd.to_numeric(df['completions'], errors='coerce')
    aff_frames.append(df[df['ons_code'].notna()])

t_aff = pd.concat(aff_frames, ignore_index=True)
t_aff = t_aff[['ons_code', 'authority_name', 'region_code', 'region_name', 'year', 'tenure', 'completions']]

print(f'Shape: {t_aff.shape}  |  Tenures: {t_aff["tenure"].unique().tolist()}')

/var/folders/jf/p4xfj7kx3_l23qw5d2zt9zgm0000gn/T/ipykernel_30382/3022428984.py:8: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df.replace(regex={r'^\[.*\]$': float('nan')})


/var/folders/jf/p4xfj7kx3_l23qw5d2zt9zgm0000gn/T/ipykernel_30382/3022428984.py:8: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df.replace(regex={r'^\[.*\]$': float('nan')})


/var/folders/jf/p4xfj7kx3_l23qw5d2zt9zgm0000gn/T/ipykernel_30382/3022428984.py:8: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df.replace(regex={r'^\[.*\]$': float('nan')})


Shape: (36444, 7)  |  Tenures: ['total_affordable', 'social_rent', 'affordable_rent', 'shared_ownership']


/var/folders/jf/p4xfj7kx3_l23qw5d2zt9zgm0000gn/T/ipykernel_30382/3022428984.py:8: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df.replace(regex={r'^\[.*\]$': float('nan')})


In [12]:
# Huntingdonshire by tenure, recent years
(
    t_aff[t_aff['ons_code'] == 'E07000011']
    [t_aff['year'] >= '2019']
    .pivot_table(index='year', columns='tenure', values='completions', aggfunc='sum')
)

/var/folders/jf/p4xfj7kx3_l23qw5d2zt9zgm0000gn/T/ipykernel_30382/1635148109.py:3: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  t_aff[t_aff['ons_code'] == 'E07000011']


tenure,affordable_rent,shared_ownership,social_rent,total_affordable
year,,,,
2019-20,404.0,175.0,0.0,588.0
2020-21,213.0,149.0,10.0,395.0
2021-22,193.0,154.0,15.0,384.0
2022-23,274.0,136.0,49.0,459.0
2023-24,236.0,174.0,174.0,590.0
2024-25,187.0,124.0,17.0,328.0


---
## Sanity check — Huntingdonshire across tables

Three tables cover Huntingdonshire completions. They measure slightly different things so the numbers won't match exactly, but they should be in the same order of magnitude:

- Table 122 (net additions) = new builds + conversions − demolitions
- Table 253 (all completions) = new builds only
- Table 1008C (total affordable) = subset of Table 253

Expected: Table 122 ≈ Table 253, Table 1008C lower than both.

In [13]:
HDC = 'E07000011'
years = ['2020-21', '2021-22', '2022-23', '2023-24', '2024-25']

check = pd.DataFrame({
    'net_additions': t122[t122['ons_code'] == HDC].set_index('year')['net_additions'],
    'new_build_completions': t253[t253['ons_code'] == HDC].set_index('year')['all_completions'],
    'affordable_completions': (
        t_aff[(t_aff['ons_code'] == HDC) & (t_aff['tenure'] == 'total_affordable')]
        .set_index('year')['completions']
    ),
}).loc[years]

check['affordable_%'] = (check['affordable_completions'] / check['net_additions'] * 100).round(1)
check

,net_additions,new_build_completions,affordable_completions,affordable_%
year,,,,
2020-21,1132.0,830,395.0,34.9
2021-22,1055.0,890,384.0,36.4
2022-23,1250.0,1020,459.0,36.7
2023-24,1239.0,1440,590.0,47.6
2024-25,858.0,920,328.0,38.2


Figures are consistent. The affordable delivery rate is sitting at 35–48% of total net additions — notably high and worth unpacking in the EDA notebook. The 2024-25 figures are provisional and may be revised.

---
## Save

In [14]:
def save(df, csv_name, table_name):
    df.to_csv(PROCESSED / csv_name, index=False)
    to_sqlite(df, table_name)
    print(f'  {csv_name}: {len(df):,} rows')

save(t122,  'net_additions_la.csv',          'net_additions_la')
save(t244,  'completions_england.csv',        'completions_england')
save(t253,  'starts_completions_la.csv',      'starts_completions_la')
save(t118,  'net_additions_regional.csv',     'net_additions_regional')
save(t_aff, 'affordable_completions_la.csv',  'affordable_completions_la')

  net_additions_la.csv: 10,104 rows
  completions_england.csv: 80 rows
  starts_completions_la.csv: 14,553 rows
  net_additions_regional.csv: 3,170 rows


  affordable_completions_la.csv: 36,444 rows


---
## What's next

**02 — Exploratory Analysis**

- England's long-run delivery history (1946–present)
- Regional variation and the East of England picture
- Huntingdonshire against comparable districts
- Tenure shift in affordable housing over time
- The starts vs completions pipeline story